# Notebook 02: Target Pairs and Horizon EDA

**Purpose:** Understand the structure of the 424 prediction targets:
- Which assets and asset pairs appear?
- How are targets distributed across 4 forecast lags?
- What do the return distributions look like per lag?
- Where is the missingness concentrated?

Key question: **are short-lag (1–2 day) targets more predictable than long-lag (3–4 day) targets?**

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_labels, load_target_pairs, get_labels_for_lag

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

labels = load_labels()
pairs  = load_target_pairs()
print('Loaded.')

## 1. Target pair structure

In [ ]:
# Classify targets: single asset vs pairwise spread
pairs['is_spread'] = pairs['pair'].str.contains(' - ')

print('Target type breakdown:')
print(pairs['is_spread'].value_counts().rename({True: 'Spread (pair)', False: 'Single asset'}))
print()
print('By lag:')
print(pairs.groupby('lag')['is_spread'].value_counts().unstack())

In [ ]:
# Which asset categories appear in target pairs?
def extract_assets(pair_str):
    return [a.strip() for a in pair_str.split(' - ')]

all_assets = []
for p in pairs['pair']:
    all_assets.extend(extract_assets(p))

asset_series = pd.Series(all_assets)

def categorise(asset):
    if asset.startswith('LME'): return 'LME'
    if asset.startswith('JPX'): return 'JPX'
    if asset.startswith('FX'):  return 'FX'
    if asset.startswith('US'):  return 'US Equity'
    return 'Other'

print('Asset category distribution in targets:')
print(asset_series.map(categorise).value_counts())

## 2. Return distributions by lag

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=False)

for i, lag in enumerate([1, 2, 3, 4]):
    ax = axes[i // 2][i % 2]
    lag_df = get_labels_for_lag(labels, pairs, lag)
    target_cols = [c for c in lag_df.columns if c != 'date_id']
    all_values = lag_df[target_cols].values.ravel()
    valid = all_values[~np.isnan(all_values)]
    
    ax.hist(valid, bins=80, alpha=0.7, density=True)
    ax.set_title(f'Lag {lag} return distribution  (n={len(valid):,})')
    ax.set_xlabel('Target value')
    ax.axvline(0, color='red', lw=0.8, linestyle='--')
    
    print(f'Lag {lag}: mean={valid.mean():.4f}  std={valid.std():.4f}  '
          f'skew={pd.Series(valid).skew():.3f}')

plt.tight_layout()
plt.suptitle('Target return distributions by lag', y=1.01, fontsize=13)
plt.show()

## 3. Target missingness by lag

In [ ]:
print('Missing values by lag (% of values):')
for lag in (1, 2, 3, 4):
    lag_df = get_labels_for_lag(labels, pairs, lag)
    target_cols = [c for c in lag_df.columns if c != 'date_id']
    n_total   = lag_df[target_cols].size
    n_missing = lag_df[target_cols].isnull().sum().sum()
    print(f'  lag={lag}  {n_missing:,} / {n_total:,}  ({100 * n_missing / n_total:.1f}%)')

## 4. Autocorrelation of target values over time

In [ ]:
# Check lag-1 autocorrelation of each target, averaged across targets
lag1_df = get_labels_for_lag(labels, pairs, lag=1)
target_cols = [c for c in lag1_df.columns if c != 'date_id']

autocorrs = [
    lag1_df[col].autocorr(lag=1)
    for col in target_cols
]
autocorrs = pd.Series(autocorrs, name='autocorr_lag1').dropna()

print(f'Lag-1 autocorrelation of targets (lag=1 horizon):')
print(autocorrs.describe().round(4))

plt.figure(figsize=(8, 3))
autocorrs.hist(bins=40)
plt.axvline(0, color='red', lw=0.8, linestyle='--')
plt.title('Distribution of lag-1 autocorrelation across targets')
plt.xlabel('Autocorrelation coefficient')
plt.tight_layout()
plt.show()

## 5. Key observations

Fill in after running:

- Spread vs single-asset targets: **TBD**
- Dominant asset categories in targets: **TBD**
- Mean return near 0 at all lags: **TBD**
- Distribution shape (fat tails?): **TBD**
- Missingness pattern by lag: **TBD**
- Average lag-1 autocorrelation (mean-reversion or momentum?): **TBD**